# 📓 Notebook 05 — LLM-as-Judge Evaluation & Calibration Study

**LUMINA Project** · *Mistral LLM-as-Judge: 4-axis scoring with human correlation analysis*

This notebook demonstrates:
1. Live LLM-as-Judge scoring on real SEC EDGAR excerpts
2. Human-vs-Judge correlation analysis (calibration)
3. Reward distribution per agent
4. Prompt version A/B comparison
5. Score reliability (inter-run variance)
6. Integration of judge scores as PPO rewards

---

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

os.environ['MISTRAL_API_KEY'] = 'hdd30lQaVEcAv9WWugWTh1nOxoO1a3hH'
print('Environment ready ✓')

## 1. Sample Evaluation Dataset — SEC EDGAR + DocVQA

In [ ]:
# Curated sample pairs: (source, agent_output, agent_name, human_score)
# human_score: crowd-sourced rating 0-1 (used for calibration only)

eval_pairs = [
    {
        'chunk_id': 'aapl_10k_2022_rev',
        'source_excerpt': 'Apple Inc. reported total net revenues of $394.3 billion for fiscal year 2022, compared to $365.8 billion in fiscal 2021, representing an increase of 7.8 percent. Net income was $99.8 billion.',
        'agent_output': 'Apple\'s FY2022 revenue totalled $394.3 billion, up 7.8% from $365.8 billion in FY2021. Net income was $99.8 billion.',
        'agent_name': 'summariser',
        'task_type': 'summarisation',
        'human_score': 0.92
    },
    {
        'chunk_id': 'aapl_10k_2022_iphone',
        'source_excerpt': 'iPhone revenue increased 6.6% year over year to $205.5 billion. The increase reflected higher net sales of iPhone 14, partially offset by lower net sales of other iPhone models.',
        'agent_output': 'iPhone revenues grew to $205.5 billion in FY2022.',
        'agent_name': 'summariser',
        'task_type': 'summarisation',
        'human_score': 0.74   # misses the reason (partially incomplete)
    },
    {
        'chunk_id': 'msft_10k_azure',
        'source_excerpt': 'Azure and other cloud services revenue increased 28% and 27% in constant currency. Server products and cloud services revenue increased 22%.',
        'agent_output': 'Organisations: Microsoft Azure. Financial Values: 28%, 27%, 22% growth.',
        'agent_name': 'ner',
        'task_type': 'token_classification',
        'human_score': 0.80
    },
    {
        'chunk_id': 'msft_10k_hallucinated',
        'source_excerpt': 'Azure and other cloud services revenue increased 28% in constant currency.',
        'agent_output': 'Azure revenue grew by 35%, driven primarily by enterprise AI adoption and new data centre openings in Southeast Asia.',  # hallucinated facts
        'agent_name': 'document_qa',
        'task_type': 'document_qa',
        'human_score': 0.15  # severe hallucination
    },
    {
        'chunk_id': 'meta_10k_dau',
        'source_excerpt': 'Daily active users (DAU) were 1.98 billion on average for December 2022, an increase of 4% year-over-year. Monthly active users (MAU) were 2.96 billion as of December 31, 2022.',
        'agent_output': 'Meta reported 1.98 billion daily active users in December 2022 (+4% YoY) and 2.96 billion monthly active users.',
        'agent_name': 'document_qa',
        'task_type': 'document_qa',
        'human_score': 0.96
    },
    {
        'chunk_id': 'googl_10k_ads',
        'source_excerpt': 'Google advertising revenues were $224.5 billion in 2022, compared to $209.5 billion in 2021.',
        'agent_output': 'Google\'s ad revenues reached $224.5 billion in 2022 versus $209.5 billion in 2021.',
        'agent_name': 'summariser',
        'task_type': 'summarisation',
        'human_score': 0.91
    },
    {
        'chunk_id': 'amzn_10k_aws',
        'source_excerpt': 'AWS net sales were $80.1 billion in 2022, representing 16% of consolidated net sales. AWS operating income was $22.8 billion.',
        'agent_output': 'AWS: Net sales $80.1B (16% of total). Operating income: $22.8B. Entities: Amazon Web Services. Dates: 2022.',
        'agent_name': 'ner',
        'task_type': 'token_classification',
        'human_score': 0.87
    },
    {
        'chunk_id': 'tsla_10k_revenue',
        'source_excerpt': 'Total revenues in 2022 were $81.5 billion, an increase of $27.6 billion, compared to total revenues of $53.8 billion in 2021.',
        'agent_output': 'Tesla revenue was approximately $80 billion in 2022.',  # imprecise
        'agent_name': 'document_qa',
        'task_type': 'document_qa',
        'human_score': 0.61
    },
]

print(f'Evaluation dataset: {len(eval_pairs)} source-output pairs')
pd.DataFrame(eval_pairs)[['chunk_id','agent_name','task_type','human_score']].style.background_gradient(subset=['human_score'], cmap='RdYlGn')

## 2. Run LLM-as-Judge Scoring (Live Mistral API)

In [ ]:
from evaluation.llm_judge import LLMJudge

judge = LLMJudge()
results = []

for pair in tqdm(eval_pairs, desc='LLM-Judge scoring'):
    score = judge.score(
        source_excerpt=pair['source_excerpt'],
        agent_output=pair['agent_output'],
        agent_name=pair['agent_name'],
        chunk_id=pair['chunk_id'],
        task_type=pair['task_type'],
    )
    results.append({
        'chunk_id':       pair['chunk_id'],
        'agent_name':     pair['agent_name'],
        'human_score':    pair['human_score'],
        'accuracy':       score.accuracy,
        'faithfulness':   score.faithfulness,
        'completeness':   score.completeness,
        'coherence':      score.coherence,
        'composite':      score.composite,
        'reward':         score.reward,
        'reasoning':      score.reasoning,
        'latency_ms':     score.latency_ms,
    })

df = pd.DataFrame(results)
print('Scoring complete ✓')
df[['chunk_id','agent_name','human_score','composite','reward','faithfulness']].round(3)

## 3. Human-vs-Judge Correlation (Calibration)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LLM-as-Judge Calibration: Human vs. Mistral Composite Score', fontweight='bold')

# Scatter + regression
r, p = stats.pearsonr(df['human_score'], df['composite'])
tau, p_tau = stats.kendalltau(df['human_score'], df['composite'])

ax = axes[0]
colors_by_agent = {'summariser': '#7F77DD', 'ner': '#1D9E75', 'document_qa': '#D85A30'}
for agent, grp in df.groupby('agent_name'):
    ax.scatter(grp['human_score'], grp['composite'],
               label=agent, color=colors_by_agent.get(agent, 'gray'),
               s=100, zorder=3)

# Add diagonal
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect calibration')

# Regression line
m, b = np.polyfit(df['human_score'], df['composite'], 1)
x_range = np.linspace(df['human_score'].min(), df['human_score'].max(), 50)
ax.plot(x_range, m * x_range + b, color='gray', linestyle='--', alpha=0.6)

ax.set_xlabel('Human Score'); ax.set_ylabel('LLM-Judge Composite')
ax.set_title(f'Pearson r={r:.3f} (p={p:.3f})\nKendall τ={tau:.3f}')
ax.legend(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(alpha=0.3)

# 4-axis radar chart
from matplotlib.patches import FancyArrowPatch
axes_labels = ['Accuracy', 'Faithfulness', 'Completeness', 'Coherence']
means = [df['accuracy'].mean(), df['faithfulness'].mean(),
         df['completeness'].mean(), df['coherence'].mean()]
human_proxy = [df['human_score'].mean()] * 4  # rough proxy

x = np.arange(len(axes_labels))
w = 0.35
axes[1].bar(x - w/2, means, w, label='LLM-Judge', color='#7F77DD', alpha=0.85)
axes[1].bar(x + w/2, human_proxy, w, label='Human (mean)', color='#1D9E75', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(axes_labels)
axes[1].set_ylim(0, 1); axes[1].set_ylabel('Score')
axes[1].set_title('Mean Score by Axis: Judge vs. Human')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
for xi, v in zip(x - w/2, means):
    axes[1].text(xi, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/judge_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Pearson r={r:.3f} | Kendall τ={tau:.3f}')
print(f'Mean judge latency: {df["latency_ms"].mean():.0f}ms')

## 4. Faithfulness as Hallucination Detector

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Judge Axis Breakdown by Agent', fontweight='bold')

axes_names = ['accuracy', 'faithfulness', 'completeness']
palettes = ['Purples', 'Greens', 'Oranges']

for ax, axis_name, pal in zip(axes, axes_names, palettes):
    agent_scores = df.groupby('agent_name')[axis_name].apply(list)
    agents = list(agent_scores.index)
    scores_list = [agent_scores[a] for a in agents]
    bp = ax.boxplot(scores_list, labels=agents, patch_artist=True)
    palette = plt.cm.get_cmap(pal)
    for patch, color in zip(bp['boxes'], [palette(0.5), palette(0.6), palette(0.7)]):
        patch.set_facecolor(color)
    ax.axhline(y=0.65, linestyle='--', color='red', alpha=0.5, label='Quality gate')
    ax.set_title(axis_name.capitalize()); ax.set_ylabel('Score'); ax.set_ylim(0, 1)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/judge_axis_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

# Highlight the hallucinated example
hallucinated = df[df['chunk_id'] == 'msft_10k_hallucinated']
print('\nHallucination detection:')
print(f"  Human score:      {hallucinated['human_score'].values[0]:.2f}")
print(f"  Judge composite:  {hallucinated['composite'].values[0]:.3f}")
print(f"  Judge faithfulness: {hallucinated['faithfulness'].values[0]:.3f} ← catches hallucination")
print(f"  Reasoning: {hallucinated['reasoning'].values[0]}")

## 5. Reward Distribution — What the RL Router Sees

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward distribution
axes[0].hist(df['reward'], bins=15, color='#7F77DD', edgecolor='white', alpha=0.85)
axes[0].axvline(x=0, color='red', linestyle='--', label='Neutral (r=0)')
axes[0].axvline(x=df['reward'].mean(), color='green', linestyle='--',
                label=f'Mean r={df["reward"].mean():.3f}')
axes[0].set_title('RL Reward Distribution (from LLM-Judge)')
axes[0].set_xlabel('Reward ∈ [-1, 1]'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Composite score vs human score error
df['error'] = df['composite'] - df['human_score']
df_sorted = df.sort_values('human_score')

axes[1].bar(range(len(df_sorted)), df_sorted['error'],
            color=['#D85A30' if e < 0 else '#1D9E75' for e in df_sorted['error']],
            alpha=0.85)
axes[1].axhline(y=0, color='black', linewidth=0.8)
axes[1].set_xticks(range(len(df_sorted)))
axes[1].set_xticklabels(df_sorted['chunk_id'].str[:15], rotation=45, ha='right', fontsize=8)
axes[1].set_title('Judge Error vs. Human (Judge − Human)')
axes[1].set_ylabel('Score Difference')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/reward_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

mae = df['error'].abs().mean()
print(f'Mean Absolute Error (judge vs human): {mae:.4f}')
print(f'Std of error: {df["error"].std():.4f}')

## 6. Score Reliability — Inter-Run Variance

Run the judge 3× on the same pair to measure consistency (low variance = reliable reward signal).

In [ ]:
# Test reliability on first 3 pairs
N_RUNS = 3
reliability_results = []

for pair in eval_pairs[:3]:
    run_scores = []
    for _ in range(N_RUNS):
        s = judge.score(
            source_excerpt=pair['source_excerpt'],
            agent_output=pair['agent_output'],
            agent_name=pair['agent_name'],
            chunk_id=pair['chunk_id'],
        )
        run_scores.append(s.composite)

    reliability_results.append({
        'chunk_id': pair['chunk_id'][:20],
        'runs': run_scores,
        'mean': np.mean(run_scores),
        'std': np.std(run_scores),
        'cv': np.std(run_scores) / (np.mean(run_scores) + 1e-8),
    })

rel_df = pd.DataFrame(reliability_results)
print('\nJudge Reliability (3 runs per pair):')
print(rel_df[['chunk_id','mean','std','cv']].round(4).to_string(index=False))
print(f'\nMean CV across pairs: {rel_df["cv"].mean():.4f} (lower = more consistent)')
print('A CV < 0.05 indicates the judge is stable enough for RL reward signals ✓')

## 7. Session Summary & MLflow Export

In [ ]:
summary = judge.session_summary()
print('\n=== LLM-Judge Session Summary ===')
for k, v in summary.items():
    if isinstance(v, dict):
        print(f'  {k}:')
        for k2, v2 in v.items():
            print(f'    {k2}: {v2}')
    else:
        print(f'  {k}: {v}')

# Export evaluation history to JSON
os.makedirs('../outputs/eval', exist_ok=True)
judge.export_history('../outputs/eval/judge_history_nb05.json')
print('\n✓ Eval history exported → outputs/eval/judge_history_nb05.json')

# Save score CSV for use in other notebooks
df.to_csv('../outputs/eval/llm_judge_scores.csv', index=False)
print('✓ Scores CSV saved → outputs/eval/llm_judge_scores.csv')